# 03 - Model selection, Stage 2 (instances) and Stage 3 (FIGO) - CPU runtime

UMD has **no FIGO labels**: the FIGO types below are derived from masks by geometric rules. We validate
(a) the rule-derived distribution on the expert masks against published clinical frequencies, and
(b) each model by how well its masks reproduce the expert-mask FIGO calls. A radiologist spot-check of a
small subset is still needed before any accuracy claim. Decision support only.

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/acm-research-f26/Radii.git'  # public; if made private see README (token in Colab Secrets)
REPO = '/content/figo-repo'

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt

In [ ]:
# 1) rank Stage 1 models on cavity/wall boundary, cavity+wall Dice and downstream FIGO agreement
!python -m figo.select_models --results-dir {DRIVE}/results

In [ ]:
# 2) FIGO types from the EXPERT masks on all 300 cases, then compare with published frequencies
!python -m figo.stage3 --seg-dir {DRIVE}/data/clean --suffix _seg --out {DRIVE}/results/figo_expert_all.csv
!python -m figo.figo_eval distribution --csv {DRIVE}/results/figo_expert_all.csv

In [ ]:
# 3) FIGO from a selected model's predicted masks (test set) + distribution
BEST = 'nnunet_resenc'   # set after step 1
!python -m figo.stage3 --seg-dir {DRIVE}/preds/{BEST} --out {DRIVE}/results/figo_{BEST}_test.csv
!python -m figo.figo_eval distribution --csv {DRIVE}/results/figo_{BEST}_test.csv

In [ ]:
# 4) per-fibroid report with rationale and caution flags (what a radiologist would review)
import csv
rows = list(csv.DictReader(open(f'{DRIVE}/results/figo_{BEST}_test.csv')))
for r in rows[:15]:
    print(f"{r['case_id']} #{r['fibroid_id']}: FIGO {r['figo']:4s} {r['volume_ml']:>7s} ml  {r['rationale']}"
          + (f"  [{r['flags']}]" if r['flags'] else ''))